## **Análisis componentes para crear el nodo maestro `hadoop-master`**

### **`Dockerfile`**

**`FROM ubuntu:latest`**: Esta línea indica que la imagen que se va a construir se basará en la imagen base de Ubuntu, específicamente la versión "latest" (última). En Docker, una imagen base proporciona el sistema de archivos inicial y los binarios necesarios para ejecutar una aplicación. En este caso, la imagen base es Ubuntu, lo que significa que la imagen de Docker contendrá un sistema de archivos similar al de Ubuntu.

**`MAINTAINER Alfonso Perez Lino`**: Esta línea se utiliza para especificar el nombre del mantenedor o creador de la imagen. En versiones anteriores de Docker, esta línea era importante para proporcionar información sobre quién mantenía o creaba la imagen. Sin embargo, a partir de Docker 1.13, la instrucción **MAINTAINER** está obsoleta en favor de las etiquetas **LABEL**. Ahora, la información del mantenedor se suele agregar como una etiqueta de metadatos en lugar de utilizar la instrucción **MAINTAINER**.

**`LABEL creador="Alfonso Perez Lino"`**: Esta línea agrega metadatos a la imagen resultante. Los metadatos proporcionan información adicional sobre la imagen, como el nombre del creador, la versión, la descripción, etc. En este caso, se está utilizando la etiqueta creador para indicar el nombre del creador de la imagen. Esta información puede ser útil para otros usuarios que trabajen con la imagen, ya que proporciona contexto sobre su origen y su propósito.

In [ ]:
FROM hadoop-cluster-base
LABEL Alfonso Perez Lino

___

**`USER root`**: Esta línea indica que se cambiará al usuario root dentro del entorno de la imagen Docker. El usuario root es el superusuario en sistemas basados en Unix y tiene privilegios completos para realizar cualquier acción en el sistema. Al cambiar al usuario root, se obtienen permisos y privilegios completos dentro del entorno de la imagen, lo que permite ejecutar comandos que pueden requerir permisos elevados, como instalar paquetes, configurar el sistema, y realizar otras tareas administrativas.

In [ ]:
# Cambiamos al usuario root
USER root

___

**`WORKDIR ${BASE_DIR}`**: Este comando establece el directorio de trabajo actual dentro del contenedor. **WORKDIR** se utiliza para cambiar el directorio de trabajo en el contenedor para el resto de los comandos en el Dockerfile. En este caso, se establece el directorio de trabajo como el valor de la variable de entorno **BASE_DIR**, que es **/opt/bd**. Esto significa que todos los comandos posteriores en el Dockerfile se ejecutarán en el contexto de este directorio dentro del contenedor. Esto puede ser útil para organizar el espacio de trabajo y las operaciones del contenedor.

In [ ]:
WORKDIR ${BASE_DIR}

___

`echo "$LOG_TAG Crear directorio namenode"`: Este comando simplemente imprime un mensaje en la salida estándar que indica que se está creando el directorio del nodo maestro (namenode). `$LOG_TAG` es una variable de entorno que probablemente contenga una etiqueta o identificador para el registro, como un nivel de gravedad o un nombre de aplicación.

`mkdir -p ${BASE_DIR}/hdfs/namenode`: Este comando crea un directorio llamado `namenode` dentro de la ruta especificada por `${BASE_DIR}/hdfs/`. La opción `-p` le dice al comando `mkdir` que cree todos los directorios necesarios en la ruta proporcionada si alguno de ellos aún no existe. `${BASE_DIR}` es una variable de entorno que apunta al directorio base **/opt/bd** donde se almacenarán los datos relacionados con Hadoop en el contenedor.

In [ ]:
#Creamos los directorios para los namenodes, los datanodes y los logs de Hadoop
RUN echo "$LOG_TAG Crear directorio namenode" && \
    mkdir -p ${BASE_DIR}/hdfs/namenode 

___

Este fragmento de Dockerfile está copiando archivos de configuración y scripts desde el directorio `Config-files` de tu sistema local al directorio `/opt/bd/hadoop/etc/hadoop` dentro del contenedor.

Aquí está lo que hace cada línea:

`COPY Config-files/core-site.xml ${HADOOP_CONF_DIR}/core-site.xml`: Copia el archivo `core-site.xml` desde el directorio `Config-files` de tu sistema local al directorio de configuración de Hadoop dentro del contenedor (`${HADOOP_CONF_DIR}/core-site.xml`).

`COPY Config-files/hdfs-site.xml ${HADOOP_CONF_DIR}/hdfs-site.xml`: Copia el archivo `hdfs-site.xml` de manera similar al anterior.

`COPY Config-files/yarn-site.xml ${HADOOP_CONF_DIR}/yarn-site.xml`: Copia el archivo `yarn-site.xml` de manera similar al anterior.

`COPY Config-files/mapred-site.xml ${HADOOP_CONF_DIR}/mapred-site.xml`: Copia el archivo `mapred-site.xml` de manera similar al anterior.

Estos comandos garantizan que los archivos de configuración necesarios para Hadoop se encuentren en el lugar correcto dentro del contenedor para que Hadoop pueda utilizarlos.

In [ ]:
#Copiamos los archivos de configuración que tenemos en el directorio "Config-files" de nuestro PC al 
#directorio /opt/bd/hadoop/etc/hadoop del contenedor.
RUN echo "$LOG_TAG Copiando archivos de configuración y script de inicio"
COPY Config-files/core-site.xml ${HADOOP_CONF_DIR}/core-site.xml
COPY Config-files/hdfs-site.xml ${HADOOP_CONF_DIR}/hdfs-site.xml
COPY Config-files/yarn-site.xml ${HADOOP_CONF_DIR}/yarn-site.xml
COPY Config-files/mapred-site.xml ${HADOOP_CONF_DIR}/mapred-site.xml

___

Este fragmento de Dockerfile está copiando dos scripts desde el directorio `Config-files` de tu sistema local al directorio `/opt/bd` dentro del contenedor.

Aquí está lo que hace cada línea:

1. `COPY Config-files/start-daemons.sh ${BASE_DIR}/start-daemons.sh`: Copia el script `start-daemons.sh` desde el directorio `Config-files` de tu sistema local al directorio `/opt/bd` dentro del contenedor con el nombre `start-daemons.sh`.

2. `COPY Config-files/start-services.sh ${BASE_DIR}/start-services.sh`: Copia el script `start-services.sh` de manera similar al anterior, con el nombre `start-services.sh`.

Estos comandos garantizan que los scripts necesarios para iniciar los daemons de Hadoop y el servicio SSH se encuentren en el lugar correcto dentro del contenedor.

In [ ]:
# Script de inicio de daemons y SSH
COPY Config-files/start-daemons.sh ${BASE_DIR}/start-daemons.sh
COPY Config-files/start-services.sh ${BASE_DIR}/start-services.sh

___

Este fragmento de Dockerfile establece los permisos de ejecución para los scripts `start-daemons.sh` y `start-services.sh` dentro del contenedor. 

Aquí está lo que hace cada línea:

1. `chmod +x ${BASE_DIR}/start-daemons.sh`: Otorga permisos de ejecución al script `start-daemons.sh` ubicado en el directorio `${BASE_DIR}` dentro del contenedor.

2. `chmod +x ${BASE_DIR}/start-services.sh`: Similar al anterior, esta línea otorga permisos de ejecución al script `start-services.sh` ubicado en el directorio `${BASE_DIR}` dentro del contenedor. 

Esto asegura que los scripts sean ejecutables dentro del contenedor.

In [ ]:
# Establecemos los permisos
RUN echo "$LOG_TAG Cambiar permisos de ejecución" && \
    chmod +x ${BASE_DIR}/start-daemons.sh && \
    chmod +x ${BASE_DIR}/start-services.sh

___

Este fragmento de código en un Dockerfile ejecuta un comando para formatear el sistema de archivos de Hadoop. Aquí está el desglose:

- `RUN`: Esta instrucción de Dockerfile ejecuta comandos durante el proceso de construcción de la imagen del contenedor.

- `echo "$LOG_TAG Formatear el sistema de archivos de hadoop"`: Esto simplemente imprime un mensaje en la salida estándar durante la construcción de la imagen. El contenido de $LOG_TAG no está definido aquí, pero podría ser una variable de entorno definida previamente en el Dockerfile.

- `${HADOOP_HOME}/bin/hdfs namenode -format -nonInteractive 2> /dev/null`: Esta es la parte importante. Ejecuta el comando **hdfs namenode -format -nonInteractive** que formatea el namenode del sistema de archivos de Hadoop. Los argumentos **-format -nonInteractive** indican que el formateo debe realizarse en modo no interactivo, lo que significa que no se solicitará confirmación al usuario durante el proceso. El **2> /dev/null** redirige los mensajes de error a la nada (**/dev/null**), lo que significa que los mensajes de error se descartan y no se mostrarán durante la construcción de la imagen.

Este comando es crítico, ya que formatea el sistema de archivos de Hadoop y debe realizarse con precaución, ya que eliminará todos los datos existentes en el sistema de archivos. Es común realizar esta operación durante la configuración inicial de un clúster de Hadoop.

In [ ]:
#Formateamos el sistema de archivos de hadoop
RUN echo "$LOG_TAG Formatear el sistema de archivos de hadoop" && \
    ${HADOOP_HOME}/bin/hdfs namenode -format -nonInteractive 2> /dev/null

___

Este fragmento del Dockerfile expone los puertos utilizados por NameNode y ResourceManager en el contenedor.

- `EXPOSE 9870 9871`: Estos puertos se utilizan para el NameNode. El puerto 9870 se utiliza para la interfaz web del NameNode y el puerto 9871 se utiliza para la comunicación interna.
  
- `EXPOSE 8088 8090`: Estos puertos se utilizan para el ResourceManager. El puerto 8088 se utiliza para la interfaz web del ResourceManager y el puerto 8090 se utiliza para la comunicación interna.

In [ ]:
# Puertos NameNode
EXPOSE 9870 9871

# Puertos ResourceManager
EXPOSE 8088 8090  

___

Este fragmento de Dockerfile establece el directorio de trabajo como el directorio raíz de Hadoop especificado por la variable de entorno `${HADOOP_HOME}`. Luego, se ejecuta el script `start-services.sh` como el comando predeterminado cuando se inicia el contenedor. Este script se utiliza para iniciar los servicios necesarios de Hadoop, como SSH, NameNode, DataNode, ResourceManager, etc.

In [ ]:
WORKDIR ${HADOOP_HOME}

RUN echo "$LOG_TAG Iniciando servicios"
CMD ["/opt/bd/start-services.sh"]

### **Config-files/`hdfs-site.xml`**

El archivo **hdfs-site-namenode.xml** en Hadoop contiene configuraciones específicas para el NameNode del sistema de archivos distribuido de Hadoop (`HDFS`). Cada propiedad en este archivo configura un aspecto diferente del funcionamiento del NameNode. Vamos a desglosar y explicar los componentes proporcionados.

**Estructura General**

El archivo está envuelto en una etiqueta `<configuration>`, dentro de la cual hay varias etiquetas `<property>`. Cada `<property>` define una propiedad de configuración con sus respectivas etiquetas `<name>`, `<value>`, y opcionalmente `<final>`.

**Componentes del `hdfs-site.xml`**

**1. Configuración de `dfs.namenode.name.dir`**

```xml
                        <property>
                            <name>dfs.namenode.name.dir</name>
                            <value>/opt/bd/hdfs/namenode</value>
                            <description>NameNode directory for namespace and transaction logs storage.</description>
                        </property>
```
**`<name>dfs.namenode.name.dir</name>`**: Esta propiedad define la ubicación en el sistema de archivos local donde el NameNode debe almacenar los metadatos. **dfs.namenode.name.dir** es el nombre de la propiedad.

**`<value>/opt/bd/hdfs/namenode</value>`**: El valor de esta propiedad es **/opt/bd/hdfs/namenode**, indicando que los metadatos del NameNode se almacenarán en el directorio **/opt/bd/hdfs/namenode** en el sistema de archivos local. Se puede configurar una lista separada por comas de directorios para replicar los metadatos en múltiples ubicaciones, proporcionando redundancia.

**`<final>true</final>`**: Esta etiqueta indica que el valor de la propiedad es final y no puede ser sobrescrito.

**2. Configuración de `dfs.replication`**

```xml
                                        <property>
                                          <!-- Block replication factor (default value 3) -->
                                          <name>dfs.replication</name>
                                          <value>3</value>
                                        </property>
``` 
**`<name>dfs.replication</name>`**: Esta propiedad define el factor de replicación de bloques en HDFS. **dfs.replication** es el nombre de la propiedad.

**`<value>3</value>`**: El valor de esta propiedad es **3**, lo que significa que cada bloque de datos se replicará en tres nodos diferentes en el clúster. Este es el valor predeterminado para proporcionar redundancia y alta disponibilidad.

**`<final>true</final>`**: Esta etiqueta indica que el valor de la propiedad es final y no puede ser sobrescrito por configuraciones posteriores.

**3. Configuración de `dfs.blocksize`**

```xml
                                          <property>
                                            <!-- Block size (default value 128m) -->
                                            <name>dfs.blocksize</name>
                                            <value>64m</value>
                                          </property>
```
**`<name>dfs.blocksize</name>`**: Esta propiedad especifica el tamaño de los bloques en HDFS. **dfs.blocksize** es el nombre de la propiedad.

**`<value>64m</value>`**: El valor de esta propiedad es **64m**, lo que significa que el tamaño de cada bloque es de 64 megabytes. El valor predeterminado suele ser 128 megabytes, pero aquí se ha configurado a 64 megabytes.

**4. Configuración de `dfs.namenode.http-address`**

```xml
                                          <property>
                                            <name>dfs.namenode.http-address</name>
                                            <value>0.0.0.0:9870</value>
                                          </property>
```
**`<name>dfs.namenode.http-address</name>`**: Esta propiedad define la dirección y el puerto base donde la interfaz web del NameNode estará escuchando. **dfs.namenode.http-address** es el nombre de la propiedad.

**`<value>0.0.0.0:9870</value>`**: El valor de esta propiedad es **0.0.0.0:9870**, indicando que la interfaz web del NameNode estará accesible en el host 0.0.0.0 en el puerto 9870.

**5. Configuración de `dfs.heartbeat.interval`**

**Modificar el tiempo de deteccion de un nodo caido**

Por defecto el namenode recibe latidos de los datanode de una manera regular. Estos latidos ademas de ser una manera de que el datanode sigue operativo, tambien incluyen informacion sobre su estado. De esta manera el namenode puede tomar decisiones en base a la carga de almacenamiento o procesamiento de cada nodo. Cuando un nodo de le considera muerto, el namenode deja de asignarle operaciones de lectura o escritura y los datos que tuviera almacenados son replicados en otros nodos para mantener el numero de replicas configurado.

- `hdfs-site.xml` ====> **dfs.heartbeat.interval** ====> Por defecto, son 3 segundos

- `Por linea de comando tambien podemos ver este valor` ====> **hdfs getconf -confkey dfs.heartbeat.interval**

Que la frecuencia de los latidos sea de 3 segundos (1 latido c/ 3 segundos), no quiere decir que ante la falta de un latido durante 5 segundos, se desencadene la copia de replicas. Piensa que en un cluster grandes la copia de replicas facilmente puede suponer varios teras que hay que mover por la red con la consiguiente saturacion que ello supone. Por eso hadoop utiliza una formula que declara un nodo muerto. Es una formula bastante rara y consiste en esperar 10 latidos perdidos durante 2 periodos de tiempo consecutivos (5 min + 5 min = 10 min). Por defecto, cada periodo equivale a 300 segundos (300.000 milisegundos), es decir, 5 minutos. Con estos valores en mente y aplicando la formula, entonces, si 
durante 30 segundos, que son los 10 latidos, un nodo no responde se concedera 2 periodos de gracia que son 5 minutos y 5 minutos para ver si se recupera. En total el tiempo que pasa para declarar un nodo como muerto es de 10 minutos y medio. Un tiempo tan alto se justifica porque las consecuencias de que un nodo muera suponen una carga demasiado grande para la red y los nodos del cluster. Nosotros vamos a cambiar estos valores porque tenemos pocos datos, pocos nodos y ademas estos nodos estan conectados al mismo segmento de red. Detectar un nodo muerto no supondria mucha sobrecarga y evitaria errores en los clientes. 

En Hadoop, un nodo se considera muerto después de que haya excedido ciertos umbrales de tiempo sin enviar señales de vida al NameNode (en el caso de un DataNode) o al ResourceManager (en el caso de un NodeManager).

Para ser más específico:

1. **DataNode Dead Node Detection (Detección de nodos muertos de DataNode)**: En Hadoop, los DataNodes están continuamente enviando señales de vida al NameNode. Si un DataNode no envía señales de vida durante un cierto período de tiempo, el NameNode lo considerará muerto y lo eliminará de la lista de DataNodes activos. El tiempo de espera predeterminado para esto es de 10 minutos. Este valor se puede configurar mediante la propiedad `dfs.heartbeat.interval` en el archivo de configuración `hdfs-site.xml`.

2. **NodeManager Dead Node Detection (Detección de nodos muertos de NodeManager)**: De manera similar, en el contexto de YARN (el gestor de recursos de Hadoop), si un NodeManager no envía señales de vida al ResourceManager durante un cierto período de tiempo, se considera muerto. El tiempo de espera predeterminado es de 10 minutos también. Este valor se puede configurar mediante la propiedad `yarn.resourcemanager.nodemanagers.heartbeat-interval-ms` en el archivo de configuración `yarn-site.xml`.

Estos tiempos de espera se pueden ajustar según los requisitos específicos del clúster o las condiciones de red, pero los valores predeterminados suelen funcionar bien para la mayoría de los entornos.


```xml
                    <property>
                    <!-- Determina el intervalo de latido del datanode en segundos. Puede utilizar los 
                        siguientes sufijos (sin distinguir mayúsculas de minúsculas): ms(milis), s(sec), 
                        m(min), h(hour), d(day) para especificar el tiempo (como 2s, 2m, 1h, etc.). O 
                        proporcione el número completo en segundos (como 30 para 30 segundos). Si no se 
                        especifica ninguna unidad de tiempo, se asume que son segundos. -->
                        <name>dfs.heartbeat.interval</name>
                        <value>1</value>
                    </property>  
```
**`<name>dfs.heartbeat.interval</name>`**: **Es la frecuencia con la que el NameNode demandará latidos a todos los DataNodes**. La propiedad `dfs.heartbeat.interval` es una configuración en Hadoop que especifica el intervalo de tiempo en milisegundos entre los latidos del corazón (heartbeats) enviados por los DataNodes al NameNode. Los DataNodes envían estos latidos del corazón para informar al NameNode de que están vivos y operativos.

El propósito principal de estos latidos del corazón es permitir al NameNode detectar la desconexión o el fallo de un DataNode. Si un DataNode deja de enviar latidos del corazón durante un período más largo que el valor especificado en `dfs.heartbeat.interval`, el NameNode lo considerará como un nodo muerto y tomará medidas, como replicar los bloques de datos que estaban almacenados en ese DataNode para garantizar la tolerancia a fallos y la integridad de los datos.

Por defecto, el valor de `dfs.heartbeat.interval` suele ser de 3 segundos (3000 milisegundos). Este valor se puede ajustar según las necesidades y las características del clúster, pero modificarlo incorrectamente puede afectar el rendimiento y la capacidad de detección de fallos del sistema Hadoop.

**`<value>1</value>`**: El valor de esta propiedad es 1 segundo. Vamos a configurar para que el namenode solicite latidos cada segundo.

**6. Configuración de `dfs.namenode.heartbeat.recheck-interval`**

La propiedad `dfs.namenode.heartbeat.recheck-interval` es una configuración en Hadoop que determina el intervalo de tiempo en milisegundos durante el cual el NameNode espera antes de volver a verificar el estado de los DataNodes después de haberlos marcado como muertos. Cuando un DataNode deja de enviar latidos del corazón al NameNode, este último lo marca como muerto después de un cierto tiempo (determinado por `dfs.namenode.availability-timeout`). Luego, el NameNode espera el intervalo especificado en `dfs.namenode.heartbeat.recheck-interval` antes de volver a verificar si el DataNode ha vuelto a estar activo.

Esta propiedad es útil para evitar la sobrecarga del NameNode al verificar constantemente el estado de los DataNodes marcados como muertos. Al esperar un período antes de volver a verificar, el NameNode puede reducir la cantidad de consultas al sistema de archivos subyacente y, por lo tanto, mejorar el rendimiento general.

El valor predeterminado de `dfs.namenode.heartbeat.recheck-interval` suele ser de 5 minutos (300000 milisegundos). Este valor se puede ajustar según las necesidades y las características del clúster, pero modificarlo incorrectamente puede afectar la capacidad del sistema para detectar y recuperarse de fallos en los DataNodes.

```xml
                    <property>
                    <!-- Este tiempo decide el intervalo para comprobar si hay datanodes muertos. Con este valor 
                        y dfs.heartbeat.interval, también se calcula el intervalo para decidir si el datanode está 
                        muerto o no. La unidad de esta configuración es el milisegundo. -->
                        <name>dfs.namenode.heartbeat.recheck-interval</name>
                        <value>500</value>    
                    </property>  
```
**`<name>dfs.namenode.heartbeat.recheck-interval</name>`**: Se mide en milisegundos, le hemos puesto medio segundo, que indica uno de los dos periodos de gracia que se concede cuando se detectan mas de 10 latidos perdidos.

**`<value>500</value>`**: Vamos a configurar para que el namenode conceda 2 periodos de gracia de 500 milisegundos (0,5 segundos) uno y 500 milisegundos (0,5 segundos) otro. 

**7. Configuración de `dfs.hosts`**

Para gestionar los equipos que pertenecen al cluster debemos modificar la propeidad "`dfs.hosts`" en el namenode y como valor le pasaremos la ruta de un archivo de texto. Este archivo tendra la IP o el nombre de cada datanode en una linea.

Modificamos el archivo "`workers`" que se encuentra en la ruta **$HADOOP_HOME/etc/hadoop**.

```xml
                    <property>
                    <!-- Nombra un archivo que contiene una lista de hosts a los que se permite conectarse 
                        al namenode. Debe especificarse la ruta completa del archivo. Si el valor está vacío, 
                        todos los hosts están permitidos.  -->
                        <name>dfs.hosts</name>
                        <value>/opt/bd/hadoop/etc/hadoop/workers</value>    
                    </property> 
```
**`<name>dfs.hosts</name>`**: La propiedad `dfs.hosts` es una configuración en Hadoop que permite especificar un archivo que contiene una lista de nombres de host permitidos para operar como DataNodes en el clúster Hadoop. Solo los DataNodes cuyos nombres de host están incluidos en este archivo podrán registrarse y participar en el clúster.

Esta configuración es útil para controlar qué nodos pueden unirse al clúster y limitar el acceso a los recursos del clúster solo a los nodos autorizados. Puede ayudar a garantizar la seguridad y la integridad del clúster al prevenir que nodos no autorizados participen en el procesamiento y almacenamiento de datos.

El formato del archivo especificado en `dfs.hosts` suele ser un archivo de texto simple que contiene una lista de nombres de host, uno por línea. Los DataNodes solo podrán unirse al clúster si su nombre de host coincide exactamente con uno de los nombres de host en este archivo.

Es importante tener en cuenta que esta configuración debe usarse junto con otras medidas de seguridad, como autenticación y control de acceso, para garantizar la seguridad del clúster Hadoop.

**`<value>/opt/bd/hadoop/etc/hadoop/workers</value>`**: Ruta donde encontramos el archivo "**workers**".

### **Config-files/`mapred-site.xml`**

**1. Configuración de `mapreduce.framework.name`**

```xml
                    <property>
                    <!-- El runtime framework para ejecutar MapReduce jobs. Puede ser local, classic o yarn.
                        Por defecto es "local" -->
                        <name>mapreduce.framework.name</name>
                        <value>yarn</value>
                    </property>
```
**`<name>mapreduce.framework.name</name>`**: especifica el framework de tiempo de ejecución (runtime framework) para los MapReduce jobs. Aquí, la propiedad **mapreduce.framework.name** se establece en **yarn**, lo que indica que se utilizará el framework YARN para ejecutar trabajos MapReduce en el clúster.

**`<value>yarn</value>`**: indica que se utilizará el framework YARN para ejecutar trabajos MapReduce en el clúster.

___

**2. Configuración de `mapreduce.application.classpath`**

```xml
            <property>
            <!-- CLASSPATH para aplicaciones MR. Una lista separada por comas de entradas CLASSPATH. 
                Si mapreduce.application.framework está establecido entonces esto debe especificar 
                el classpath apropiado para ese archivo, y el nombre del archivo debe estar presente 
                en el classpath. Si mapreduce.app-submission.cross-platform es falso, se utilizará la 
                sintaxis de expansión vairable del entorno específico de la plataforma para construir 
                las entradas CLASSPATH por defecto. Para Linux: $HADOOP_MAPRED_HOME/share/hadoop/mapreduce/*, 
                $HADOOP_MAPRED_HOME/share/hadoop/mapreduce/lib/*. Para Windows: %HADOOP_MAPRED_HOME%/share/hadoop/mapreduce/*, 
                %HADOOP_MAPRED_HOME%/share/hadoop/mapreduce/lib/*. Si mapreduce.app-submission.cross-platform 
                es true, se utilizará el CLASSPATH por defecto agnóstico de plataforma para aplicaciones 
                MR: {{HADOOP_MAPRED_HOME}}/share/hadoop/mapreduce/*, {{HADOOP_MAPRED_HOME}}/share/hadoop/mapreduce/lib/* 
                El marcador de expansión de parámetros será sustituido por NodeManager en el lanzamiento del contenedor 
                en función del SO subyacente según corresponda. -->
                <name>mapreduce.application.classpath</name>
                <value>/opt/bd/hadoop/share/hadoop/mapreduce/*</value>
            </property>   
```
**`<name>mapreduce.application.classpath</name>`**: especifica el CLASSPATH para las aplicaciones MapReduce. El CLASSPATH es una lista de ubicaciones de directorios y archivos JAR que se utilizan para buscar clases Java durante la ejecución de las aplicaciones MapReduce.

**`<value>/opt/bd/hadoop/share/hadoop/mapreduce/*</value>`**:  indica que se incluirán todos los archivos en el directorio /opt/bd/hadoop/share/hadoop/mapreduce/ en el CLASSPATH para las aplicaciones MapReduce.

La ruta `${HADOOP_HOME}/share/hadoop/mapreduce/*` contiene archivos de clase y recursos necesarios para ejecutar tareas MapReduce en un clúster Hadoop.

- `${HADOOP_HOME}` se refiere al directorio de instalación de Hadoop.
- `share/hadoop/mapreduce` es la ubicación dentro de la instalación de Hadoop donde se encuentran los archivos relacionados con MapReduce.

La expresión `*` al final de la ruta indica que se están incluyendo todos los archivos y subdirectorios dentro de la carpeta `mapreduce`.

Estos archivos suelen incluir archivos JAR de bibliotecas, configuraciones específicas de MapReduce, y otros recursos necesarios para ejecutar tareas de MapReduce en un clúster Hadoop. Por ejemplo, pueden contener clases de MapReduce, archivos de configuración como `mapred-site.xml`, `core-site.xml`, etc., y otros archivos necesarios para el funcionamiento de tareas de MapReduce.

El contenido de esta ruta se agrega al classpath de las tareas de MapReduce cuando se ejecutan en el clúster Hadoop, lo que permite que las clases y recursos necesarios estén disponibles durante la ejecución de estas tareas.

Sí, en muchos casos, especificar `${HADOOP_HOME}/share/hadoop/mapreduce/*` como parte del classpath puede ser suficiente. Esto incluirá todos los archivos y subdirectorios dentro de la carpeta `mapreduce`, lo que generalmente abarca tanto `lib` como `lib-examples`, junto con cualquier otro contenido que pueda existir en esa ubicación.

Al especificar `${HADOOP_HOME}/share/hadoop/mapreduce/*`, estás asegurando que cualquier biblioteca o recurso necesario para las tareas MapReduce, incluidos los ejemplos y las demostraciones, esté disponible en el classpath. Esto es útil si estás ejecutando tareas MapReduce estándar o si estás trabajando con los ejemplos proporcionados por Hadoop.

Sin embargo, si tienes requisitos específicos y necesitas controlar exactamente qué recursos están en el classpath, o si necesitas excluir ciertos recursos, entonces puede ser útil especificar carpetas específicas como `lib` o `lib-examples` en lugar de usar un comodín `*`.

En resumen, si simplemente necesitas asegurarte de que todas las bibliotecas y recursos relevantes de MapReduce estén disponibles en el classpath, `${HADOOP_HOME}/share/hadoop/mapreduce/*` debería ser suficiente.

### **Config-files/`yarn-site.xml`**

**1. Configuración de `yarn.resourcemanager.hostname`**

```xml
                                    <property>
                                        <name>yarn.resourcemanager.hostname</name>
                                        <value>hadoop-master</value>
                                    </property>
```
**`<name>yarn.resourcemanager.hostname</name>`**: especifica el nombre de host del ResourceManager de YARN en el clúster. Dado que en el nodo master se encuentran el namenodey el resourcemanager, ambos pueden utilizar el nombre del host que es "hadoop-master".

La propiedad `yarn.resourcemanager.hostname` se utiliza para configurar la dirección del ResourceManager para que los nodos del clúster puedan comunicarse con él correctamente.

**`<value>hadoop-master</value>`**: indica que el nombre de host del ResourceManager de YARN es `hadoop-master`.

___